In [1]:
import numpy as np
import fastf1 as ff1
import pandas as pd

In [2]:
ff1.Cache.enable_cache('C:\\Users\\jifor\\Python\\F1 Prediction Project\\FF1 Cache')

In [3]:
season = 2025
schedule = ff1.get_event_schedule(season)
schedule.columns

Index(['RoundNumber', 'Country', 'Location', 'OfficialEventName', 'EventDate',
       'EventName', 'EventFormat', 'Session1', 'Session1Date',
       'Session1DateUtc', 'Session2', 'Session2Date', 'Session2DateUtc',
       'Session3', 'Session3Date', 'Session3DateUtc', 'Session4',
       'Session4Date', 'Session4DateUtc', 'Session5', 'Session5Date',
       'Session5DateUtc', 'F1ApiSupport'],
      dtype='object')

In [32]:
session_type = 'R'
rounds = [19, 20, 21, 22, 23, 24]
res_index = 'DriverNumber'
results = pd.DataFrame(columns=rounds)
results

,19,20,21,22,23,24


In [33]:
# Create a dataframe that has the resuls of the 5 races indexed by driver number in increasing order
for round in rounds:
    session = ff1.get_session(season, round, session_type)
    session.load()
    result = pd.Series(session.results.loc[:, 'ClassifiedPosition'])
    result.index = result.index.astype(int)
    result = result.sort_index()
    result = result.replace(['R', 'D'], 20)
    results[round] = result

results

core           INFO 	Loading data for United States Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '16', '44', '81', '63', '22', '27', '87', '14', '30', '18', '12', '23', '31', '6', '43', '5', '10', '55']
core           INFO 	Loading data for Mexico City Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_

,19,20,21,22,23,24
1,1,3,3,1,1,1
4,2,1,1,20,4,3
5,18,10,20,20,13,11
6,16,13,8,6,18,17
10,19,15,10,13,16,19
12,13,6,2,3,5,15
14,10,20,14,11,7,6
16,3,2,20,4,8,4
18,12,14,16,20,17,10
22,7,11,17,12,10,14


In [36]:
X = np.array(results.loc[:, rounds[:-1]])
y = np.array(results[rounds[-1]])

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

In [40]:
model = RandomForestRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
predictions

array([13.34,  7.04,  8.73, 15.56,  5.59, 14.99, 16.19])

In [42]:
from sklearn.metrics import mean_squared_error
error = mean_squared_error(y_test, predictions)
error

25.512571428571427

In [43]:
print(y_test)
print(predictions)

['9' '2' '5' '7' '1' '11' '13']
[13.34  7.04  8.73 15.56  5.59 14.99 16.19]
